# 04 Export Job — Gold → Local (CSV/Parquet)

**役割：** Gold層のデータをDatabricksからローカルに取り出し、Streamlitが読み込める形式に変換する。  
**設計原則：** Streamlit側のコード変更を最小限にする。既存の計算ロジックをGold層に移管した分だけ差し替える。

---

### このノートブックの処理フロー
```
Delta Lake (Gold) ─── Unity Catalog Table
      │ workspace.portfolio.gold_stats
      │ workspace.portfolio.gold_corr_matrix
      │ workspace.portfolio.gold_portfolio_metrics
      │
      ├── gold_stats           → stats/part-00000-xxx.parquet
      ├── gold_corr_matrix     → corr_matrix/part-00000-xxx.parquet
      └── gold_portfolio       → portfolio_metrics/part-00000-xxx.parquet
                    │
                    ▼
             Unity Catalog Volume
             /Volumes/workspace/portfolio/exports/
                    │
                    ▼
             Catalogからダウンロード
                    │
                    ▼
             streamlit_app/data/
```

### 設計判断メモ
- **Parquet形式を採用する理由**：CSVより型情報が保持される。Streamlit側でpandasが読み込む際に型変換が不要になる
- **Unity Catalog Volumeに書き出す理由**：Serverless環境ではDBFS rootへの
  アクセスが無効化されているため、Unity Catalog VolumeをExport先として使用する。
  CatalogのUIからファイルを直接ダウンロードできる
- **単一ファイルに結合する理由**：Sparkはデフォルトで複数パートファイルに分割して書き込む。Streamlitはシングルファイルを想定しているため`coalesce(1)`で結合する

## 0. ライブラリのインポート

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

spark = SparkSession.builder.getOrCreate()
print(f"Spark version: {spark.version}")

## 1. 設定値（Config）

In [0]:
# ── Gold層テーブル名（Unity Catalog） ──────────────────────────
GOLD_STATS_TABLE     = "workspace.portfolio.gold_stats"
GOLD_CORR_TABLE      = "workspace.portfolio.gold_corr_matrix"
GOLD_PORTFOLIO_TABLE = "workspace.portfolio.gold_portfolio_metrics"

# ── エクスポート先（Unity Catalog Volume） ─────────
EXPORT_BASE      = "/Volumes/workspace/portfolio/exports"
EXPORT_STATS     = f"{EXPORT_BASE}/stats"
EXPORT_CORR      = f"{EXPORT_BASE}/corr_matrix"
EXPORT_PORTFOLIO = f"{EXPORT_BASE}/portfolio_metrics"

print(f"Gold stats path     : {EXPORT_STATS}")
print(f"Export base path    : {EXPORT_BASE}")

## 2. Gold層を読み込んでエクスポート

**`coalesce(1)` を使う理由：**  
Sparkはデータを複数のパーティションに分割して並列処理する。  
そのままwriteすると `part-00000-xxx.parquet` のような複数ファイルが生成される。  
Streamlitはシングルファイルを想定しているため、`coalesce(1)`で1ファイルに結合してから書き出す。

**pandas との対応：**
```python
# pandas
df.to_parquet('stats.parquet')

# Spark（coalesce(1)で1ファイルに結合）
sdf.coalesce(1).write.parquet(path)
```

In [0]:
def export_to_volume(table_name: str, export_path: str, label: str):
    sdf = spark.read.table(table_name)
    row_count = sdf.count()

    (
        sdf
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(export_path)
    )
    print(f"✅ {label}: {row_count} 件 → {export_path}")
    return sdf

# 3テーブルをエクスポート
sdf_stats     = export_to_volume(GOLD_STATS_TABLE,     EXPORT_STATS,     "gold_stats")
sdf_corr      = export_to_volume(GOLD_CORR_TABLE,      EXPORT_CORR,      "gold_corr_matrix")
sdf_portfolio = export_to_volume(GOLD_PORTFOLIO_TABLE, EXPORT_PORTFOLIO, "gold_portfolio_metrics")

## 3. エクスポートファイルの確認

FileStoreに書き出されたファイルの一覧を確認する。  
`part-00000-xxx.parquet` という名前のファイルが各フォルダに1つずつあればOK。

In [0]:
# FileStoreのファイル一覧を確認
for path in [EXPORT_STATS, EXPORT_CORR, EXPORT_PORTFOLIO]:
    files = dbutils.fs.ls(path)
    parquet_files = [f for f in files if f.name.endswith(".parquet")]
    print(f"{path}")
    for f in parquet_files:
        print(f"  → {f.name} ({f.size:,} bytes)")
    print()

## 4. ファイルのダウンロード方法

エクスポートされたParquetファイルはUnity Catalog VolumeのUIから直接ダウンロードできる。

1. 左サイドバーの「Catalog」をクリック
2. workspace → portfolio → exports を展開
3. stats / corr_matrix / portfolio_metrics の各フォルダを開く
4. `part-00000-xxx.parquet` を右クリック → 「Download」

※ URLによるダウンロードはServerless環境のネットワーク制限により動作しない

In [0]:
# Serverless環境では非対応

# # ダウンロードURLの生成

# INSTANCE_URL = "https://dbc-04d15d9a-aa65.cloud.databricks.com"

# for export_path, label in [
#     (EXPORT_STATS,     "stats"),
#     (EXPORT_CORR,      "corr_matrix"),
#     (EXPORT_PORTFOLIO, "portfolio_metrics"),
# ]:
#     files = dbutils.fs.ls(export_path)
#     parquet_files = [f for f in files if f.name.endswith(".parquet")]
#     if parquet_files:
#         filename = parquet_files[0].name
#         relative = export_path.replace("/Volumes/workspace/portfolio/raw_data/", "")
#         url = f"{INSTANCE_URL}/files/{relative}/{filename}"
#         print(f"[{label}]")
#         print(f"  {url}")
#         print()

## 5. エクスポートデータの内容確認

Streamlitに渡す前に、各テーブルの最終確認を行う。

In [0]:
display(spark.read.table(GOLD_STATS_TABLE).orderBy("ticker"))
display(spark.read.table(GOLD_CORR_TABLE).orderBy("ticker_a", "ticker_b"))
display(spark.read.table(GOLD_PORTFOLIO_TABLE))

---

## ✅ Export Job 完了チェックリスト

- [ ] FileStoreに3つのParquetファイルが生成されている
- [ ] 各ファイルのサイズが0バイトでないことを確認した
- [ ] ダウンロードURLにアクセスしてファイルが取得できた
- [ ] ダウンロードしたファイルを `streamlit_app/data/` に配置した

---

## ローカルへの配置手順

```bash
# streamlit_app/data/ ディレクトリを作成
mkdir -p streamlit_app/data

# Catalog → workspace → portfolio → exports → stats
# → part-00000-xxx.parquet を右クリック → Download
# ダウンロードしたファイルをリネームして配置
mv ~/Downloads/part-00000-*.parquet streamlit_app/data/stats.parquet
# corr_matrix、portfolio_metrics も同様に配置
```

---

**次のステップ：** `streamlit_app/app.py` のデータ読み込み部分をGold層エクスポートファイルに差し替える